In [2]:
import supermarq as sm
import qiskit
import qiskit_superstaq as qss
from qiskit import Aer, execute


from qiskit.converters import circuit_to_dag
from qiskit.tools.visualization import dag_drawer
# dag = circuit_to_dag(circ)
# dag_drawer(dag)

from pprint import pprint
import sys
import numpy
numpy.set_printoptions(threshold=sys.maxsize)

In [3]:
unitary_backend = Aer.get_backend('unitary_simulator')

In [4]:
def remove_gates(circ, to_remove=['measure', 'reset', 'h']):
    marked = []
    for ins in circ.data:
        if ins[0].name in to_remove:
            marked.append(ins)
    marked = marked[::-1]
    for each in marked:
        circ.data.remove(each)
    return circ

In [5]:
def get_circ(cir_name, n_qubits):
    if cir_name == 'ghz':
        special = sm.benchmarks.ghz.GHZ(num_qubits=n_qubits)

    elif cir_name == 'bit_code':
        special = sm.benchmarks.bit_code.BitCode(n_qubits, 1, [1]*n_qubits)

    elif cir_name == 'ham':
        special = sm.benchmarks.hamiltonian_simulation.HamiltonianSimulation(n_qubits, 1, 1)

    elif cir_name == 'mermin_bell':
        special = sm.benchmarks.mermin_bell.MerminBell(n_qubits)

    elif cir_name == 'phase_code':
        special = sm.benchmarks.phase_code.PhaseCode(n_qubits, 1, [1]*n_qubits)

    elif cir_name == 'qaoa':
        special = sm.benchmarks.qaoa_vanilla_proxy.QAOAVanillaProxy(n_qubits)
    
    # TODO
    # elif cir_name == 'vqe':

    special_circ = sm.converters.cirq_to_qiskit(special.circuit())
    
    # removing measurements to run on unitary simulator
    special_circ.remove_final_measurements(inplace=True)
    
    ## seperate initial and final Hadamards: TODO
    special_circ = remove_gates(special_circ)
    
    return special_circ

In [6]:
uts = []
circuits = []
for c in ['ghz', 'bit_code', 'ham', 'mermin_bell', 'phase_code']:
    special_circ = get_circ(c, 3)
    circuits.append(special_circ)
    job = execute(special_circ, unitary_backend, shots=1024)
    result = job.result()
    uts.append(result.get_unitary(special_circ, 3).data) # 3 is decimal limit

In [72]:
import termcolor
def matrix_print(nparray, size=8):
    for x in nparray:
        for y in x:
            r = '{:+02.1f}'.format(y.real)
            i = '{:+02.1f}'.format(y.imag)
            if i not in ['+0.0', '-0.0'] or r not in ['+0.0', '-0.0']:
                r = termcolor.colored(r, 'red')
                i = termcolor.colored(i, 'red')
            print("(", r, ",", i, ")",sep="", end="  ")
        print(end="\n\n")

In [133]:
def matrix_print01(nparray):
    for x in nparray:
        for y in x:
            a = int(y.real or y.imag)
            if a != 0:
                print(termcolor.colored("*", 'red') ,end=" ", sep="")
            else:
                print(a, end=" ", sep="")
        print()

In [64]:
circuits[0].draw()

q_0: ──■───────
     ┌─┴─┐     
q_1: ┤ X ├──■──
     └───┘┌─┴─┐
q_2: ─────┤ X ├
          └───┘

In [73]:
# ghz
matrix_print(uts[0])

(+1.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+1.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+1.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+1.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+1.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+1.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+1.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+1.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  



In [74]:
circuits[2].draw()

┌───────────┐                                            
q_0: ┤ Rz(-3π/4) ├──■────────────────■────────────────────────
     ├───────────┤┌─┴─┐┌──────────┐┌─┴─┐                      
q_1: ┤ Rz(-3π/4) ├┤ X ├┤ Rz(-π/2) ├┤ X ├──■────────────────■──
     ├───────────┤└───┘└──────────┘└───┘┌─┴─┐┌──────────┐┌─┴─┐
q_2: ┤ Rz(-3π/4) ├──────────────────────┤ X ├┤ Rz(-π/2) ├┤ X ├
     └───────────┘                      └───┘└──────────┘└───┘

In [75]:
# ham
matrix_print(uts[2])

(+0.4,-0.9)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.4,+0.9)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.9,-0.4)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.4,-0.9)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.4,+0.9)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (-0.9,-0.4)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.4,-0.9)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (-0.4,-0.9)  



In [76]:
circuits[3].draw()

┌──────────┐                    ┌───┐┌───┐┌───┐        
q_0: ┤ Rx(-π/2) ├──■─────────■───────┤ X ├┤ X ├┤ S ├─■──────
     └──────────┘┌─┴─┐       │       └─┬─┘└─┬─┘└───┘ │ ┌───┐
q_1: ────────────┤ X ├──■────┼────■────┼────■────────■─┤ S ├
                 └───┘┌─┴─┐┌─┴─┐┌─┴─┐  │  ┌───┐        └───┘
q_2: ─────────────────┤ X ├┤ X ├┤ X ├──■──┤ S ├─────────────
                      └───┘└───┘└───┘     └───┘

In [77]:
# mermin_bell
matrix_print(uts[3])

(+0.7,+0.0)  (+0.0,+0.7)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (-0.7,+0.0)  (+0.0,+0.7)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (-0.7,+0.0)  (+0.0,+0.7)  (+0.0,+0.0)  (+0.0,+0.0)  

(-0.0,+0.0)  (-0.0,+0.0)  (+0.7,+0.0)  (+0.0,+0.7)  (-0.0,+0.0)  (-0.0,+0.0)  (-0.0,+0.0)  (-0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (-0.7,+0.0)  (+0.0,+0.7)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (-0.7,+0.0)  (-0.0,-0.7)  (+0.0,+0.0)  (+0.0,+0.0)  

(+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (+0.0,+0.0)  (-0.7,+0.0)  (-0.0,-0.7)  

(-0.7,+0.0)  (+0.0,+0.7)  (-0.0,+0.0)  (-0.0,+0.0)  (-0.0,+0.0)  (-0.0,+0.0)  (-0.0,+0.0)  (-0.0,+0.0)  



In [ ]:
# bit_code
matrix_print(uts[1])

In [ ]:
# phase_code
matrix_print(uts[4])

In [122]:
import numpy as np
np.random.seed(999999)
target_distr = np.random.rand(2)
# We now convert the random vector into a valid probability vector
target_distr /= sum(target_distr)
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
def get_var_form(params):
    qr = QuantumRegister(1, name="q")
    cr = ClassicalRegister(1, name='c')
    qc = QuantumCircuit(qr, cr)
    qc.u3(params[0], params[1], params[2], qr[0])
    qc.measure(qr, cr[0])
    return qc
from qiskit import Aer, transpile, assemble
backend = Aer.get_backend("qasm_simulator")
NUM_SHOTS = 10000
def get_probability_distribution(counts):
    output_distr = [v / NUM_SHOTS for v in counts.values()]
    if len(output_distr) == 1:
        output_distr.append(1 - output_distr[0])
    return output_distr

def objective_function(params):
    # Obtain a quantum circuit instance from the parameters
    qc = get_var_form(params)
    # Execute the quantum circuit to obtain the probability distribution associated with the current parameters
    t_qc = transpile(qc, backend)
    qobj = assemble(t_qc, shots=NUM_SHOTS)
    result = backend.run(qobj).result()
    # Obtain the counts for each measured state, and convert those counts into a probability vector
    output_distr = get_probability_distribution(result.get_counts(qc))
    # Calculate the cost as the distance between the output distribution and the target distribution
    cost = sum([np.abs(output_distr[i] - target_distr[i]) for i in range(2)])
    return cost

In [123]:
from qiskit.aqua.components.optimizers import COBYLA

# Initialize the COBYLA optimizer
optimizer = COBYLA(maxiter=500, tol=0.0001)

# Create the initial parameters (noting that our single qubit variational form has 3 parameters)
params = np.random.rand(3)
ret = optimizer.optimize(num_vars=3, objective_function=objective_function, initial_point=params)

# Obtain the output distribution using the final parameters
qc = get_var_form(ret[0])
t_qc = transpile(qc, backend)
qobj = assemble(t_qc, shots=NUM_SHOTS)
counts = backend.run(qobj).result().get_counts(qc)
output_distr = get_probability_distribution(counts)

print("Target Distribution:", target_distr)
print("Obtained Distribution:", output_distr)
print("Output Error (Manhattan Distance):", ret[1])
print("Parameters Found:", ret[0])

/tmp/ipykernel_81856/527805904.py:11: DeprecationWarning: The QuantumCircuit.u3 method is deprecated as of 0.16.0. It will be removed no earlier than 3 months after the release date. You should use QuantumCircuit.u instead, which acts identically. Alternatively, you can decompose u3 in terms of QuantumCircuit.p and QuantumCircuit.sx: u3(ϴ,φ,λ) = p(φ+π) sx p(ϴ+π) sx p(λ) (2 pulses on hardware).
  qc.u3(params[0], params[1], params[2], qr[0])


Target Distribution: [0.51357006 0.48642994]
Obtained Distribution: [0.4842, 0.5158]
Output Error (Manhattan Distance): 0.05074011873883916
Parameters Found: [ 1.53598882 -0.01728572  0.67284268]


In [126]:
qc.draw()

┌─────────────────────────────┐┌─┐
  q: ┤ U3(1.536,-0.017286,0.67284) ├┤M├
     └─────────────────────────────┘└╥┘
c: 1/════════════════════════════════╩═
                                     0

In [134]:
job = execute(special_circ, unitary_backend, shots=1024)
result = job.result()
matrix_print01(result.get_unitary(special_circ, 3).data) # 3 is decimal limit

0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 * 0 0 0 0 
0 0 0 0 0 0 0 0 0 0 0 0 0